# Stage 2 Contract Smoke Test

This notebook verifies the first Stage 2 implementation slice:

- root `src/autoresearch` package imports
- `resnet_trigger` node spec loads
- editable-scope validation works
- trial lifecycle state machine rejects invalid jumps
- formal trial record schema round-trips

It does not run the worker model or train the ResNet node.

In [1]:
from pathlib import Path
import sys

ROOT = Path("/Users/wongdowling/Documents/autoresearch_harness")
SRC = ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT

PosixPath('/Users/wongdowling/Documents/autoresearch_harness')

## 1. Load the ResNet Trigger Node Spec

Expected: the spec loads from `configs/nodes/resnet_trigger.yaml`, names the node `resnet_trigger`, allows only `train.py` edits, and uses `val_auc` as the maximized metric.

In [2]:
from autoresearch.nodes.spec import load_node_spec

spec = load_node_spec(ROOT / "configs" / "nodes" / "resnet_trigger.yaml")
spec.to_dict()

{'name': 'resnet_trigger',
 'description': 'Near-threshold detector waveform binary-classification benchmark using the ResNet_trigger node.',
 'editable_paths': ['train.py'],
 'frozen_paths': ['prepare.py',
  'program.md',
  'pyproject.toml',
  'uv.lock',
  'resnet_1d.py',
  'signal_vacuum_sum_crop_4000x8000.h5',
  'noise_traces_4000x8000.h5',
  'artifacts/',
  'data/'],
 'setup_command': 'uv sync',
 'run_command': 'uv run train.py > run.log 2>&1',
 'metric_name': 'val_auc',
 'metric_direction': 'maximize',
 'metric_parser': 'autoresearch.nodes.resnet_trigger.metric_parser:parse_val_auc',
 'acceptance_rule': 'candidate_metric > current_best_metric',
 'validity_checks': ['metric_present',
  'finite_metric',
  'editable_scope_only',
  'no_data_pipeline_modification',
  'command_exit_zero'],
 'default_budget': {'trials': 50, 'max_wall_clock_hours': 12.0},
 'expected_runtime': 'fast smoke: minutes on CPU; full campaign: overnight',
 'failure_categories': ['syntax_error',
  'runtime_error',

In [3]:
assert spec.name == "resnet_trigger"
assert spec.metric_name == "val_auc"
assert spec.metric_direction == "maximize"
assert spec.editable_paths == ("train.py",)
assert spec.default_budget.trials == 50

"node spec checks passed"

'node spec checks passed'

## 2. Validate Editable Scope

Expected: `train.py` is valid, while `prepare.py` and artifacts are rejected.

In [4]:
from autoresearch.control_plane.permissions import validate_edit_scope

valid_scope = validate_edit_scope(["train.py"], spec)
valid_scope

ScopeValidationResult(valid=True, changed_paths=('train.py',), violations=())

In [5]:
invalid_scope = validate_edit_scope(["prepare.py", "artifacts/best_model.pt"], spec)
invalid_scope

ScopeValidationResult(valid=False, changed_paths=('prepare.py', 'artifacts/best_model.pt'), violations=('prepare.py is outside editable_paths', 'prepare.py matches frozen_paths', 'artifacts/best_model.pt is outside editable_paths', 'artifacts/best_model.pt matches frozen_paths'))

In [6]:
assert valid_scope.valid is True
assert invalid_scope.valid is False
assert any("prepare.py" in violation for violation in invalid_scope.violations)
assert any("artifacts" in violation for violation in invalid_scope.violations)

"editable-scope checks passed"

'editable-scope checks passed'

## 3. Check Lifecycle State Machine

Expected: the ordered lifecycle reaches `kept`, and a direct jump from `proposed` to `executed` is rejected.

In [7]:
from autoresearch.control_plane.state_machine import (
    InvalidTransitionError,
    TrialState,
    transition,
)

state = TrialState.INITIALIZED
path = [state]

for target in [
    TrialState.PROPOSED,
    TrialState.PATCH_GENERATED,
    TrialState.SCOPE_VALIDATED,
    TrialState.EXECUTED,
    TrialState.METRIC_PARSED,
    TrialState.EVALUATED,
    TrialState.KEPT,
]:
    state = transition(state, target)
    path.append(state)

[item.value for item in path]

ImportError: cannot import name 'StrEnum' from 'enum' (/Users/wongdowling/opt/anaconda3/lib/python3.9/enum.py)

In [ ]:
try:
    transition(TrialState.PROPOSED, TrialState.EXECUTED)
except InvalidTransitionError as error:
    print("Rejected as expected:", error)
else:
    raise AssertionError("Invalid transition was not rejected")

## 4. Build and Round-Trip a Trial Record

Expected: the schema accepts a complete kept trial and can convert it to/from a plain dictionary.

In [ ]:
from autoresearch.memory.schemas import (
    ExecutionStatus,
    TrialDecision,
    TrialProvenance,
    TrialRecord,
    ValidityStatus,
)

record = TrialRecord(
    trial_id="trial-001",
    campaign_id="campaign-smoke",
    node_id="resnet_trigger",
    budget_index=1,
    timestamp_start="2026-04-28T00:00:00Z",
    timestamp_end="2026-04-28T00:01:00Z",
    manager_mode="baseline_manager",
    worker_mode="claw_style_worker",
    memory_mode="append_only_summary",
    proposal_summary="Lower learning rate",
    proposal_rationale="Prior run was unstable.",
    targeted_files=("train.py",),
    patch_ref="experiments/artifacts/trial-001/patch.diff",
    git_commit_before="abc123",
    git_commit_after="def456",
    execution_status=ExecutionStatus.SUCCESS,
    validity_status=ValidityStatus.VALID,
    failure_category=None,
    raw_log_ref="experiments/artifacts/trial-001/run.log",
    parsed_metrics={"val_auc": 0.7876},
    current_best_before=0.7799,
    delta_vs_best=0.0077,
    decision=TrialDecision.KEPT,
    decision_rationale="Validation AUC improved.",
    wall_clock_seconds=60.0,
    cumulative_budget_consumed=1,
    provenance=TrialProvenance(
        proposal_id="proposal-001",
        patch_id="patch-001",
        run_id="run-001",
        metric_id="metric-001",
        decision_id="decision-001",
    ),
)

payload = record.to_dict()
payload

In [ ]:
loaded = TrialRecord.from_mapping(payload)

assert loaded.trial_id == record.trial_id
assert loaded.parsed_metrics["val_auc"] == 0.7876
assert loaded.decision == TrialDecision.KEPT

"trial record round-trip checks passed"

## 5. Optional: Run the Formal Smoke Test

The same checks are available as a command-line test:

```bash
cd /Users/wongdowling/Documents/autoresearch_harness
PYTHONPATH=$PWD/src python3 -m unittest tests.test_stage2_contracts
```